In [ ]:
from apeGmsh import apeGmsh
from apeGmsh import Numberer, FEMData
from apeGmsh.opensees import apeSees
import numpy as np
import openseespy.opensees as ops
from pathlib import Path

In [ ]:
model_iges_path = Path(r'C:\Users\nmora\Github\pyGmsh\acad') / 'solid.iges'
assert model_iges_path.exists(), model_iges_path

m1 = apeGmsh(model_name='solid', verbose=True)
m1.begin()

m1.model.io.load_iges(file_path=model_iges_path, highest_dim_only=True)
m1.model.viewer()

In [3]:
cut_1 = m1.model.geometry.add_rectangle(
    x=-1,
    y=-1,
    z=0.5,
    dx=10,
    dy=10,
)

cut_2 = m1.model.geometry.add_rectangle(
    x=-1,
    y=-1,
    z=3.00,
    dx=10,
    dy=10,
)

m1.model.viewer()

m1.model.boolean.fragment(
    objects=[1],
    tools=[cut_1, cut_2],
    dim=3,
)

m1.model.viewer()

[Model] add_rectangle(origin=(-1,-1,0.5), size=(10,10)) → tag 67
[Model] add_rectangle(origin=(-1,-1,3.0), size=(10,10)) → tag 68

[brep_scene] Built in 0.09s  (4 actors, 365 entities)
  Mesh generate : 0.001s  (existing)
  dim0 points   : 0.032s  (116)
  dim1 curves   : 0.018s  (182)
  dim2 surfaces : 0.017s  (66)
  dim3 volumes  : 0.012s  (1)
[viewer] closed — 0 physical group(s) written, 0 picks in working set
[Model] fragment(obj=[(3, 1)], tool=[(2, 67), (2, 68)]) → tags [1, 2, 3, 4, 5, 6, 7, 73, 74]
[Model] fragment cleanup: removed 2 free surface(s)

[brep_scene] Built in 0.15s  (4 actors, 369 entities)
  Mesh generate : 0.077s  (temp coarse)
  dim0 points   : 0.029s  (116)
  dim1 curves   : 0.015s  (174)
  dim2 surfaces : 0.015s  (72)
  dim3 volumes  : 0.014s  (7)
[visibility] _rebuild_actors: 31.9ms  (4 dims, 368 hidden)
[visibility] _rebuild_actors: 45.5ms  (4 dims, 0 hidden)
[viewer] closed — 0 physical group(s) written, 0 picks in working set


In [ ]:
# --- Classify the fragmented volumes by z-center ---
import gmsh

vols = [t for _, t in gmsh.model.getEntities(3)]
print(f"Total volumes after fragment: {len(vols)}")
for v in vols:
    cz = m1.model.queries.center_of_mass(v, dim=3)[2]
    bb = m1.model.queries.bounding_box(v, dim=3)
    print(f"  vol {v}: z-center={cz:.3f}  z-range=[{bb[2]:.3f}, {bb[5]:.3f}]")

footings, columns, beams = [], [], []
for v in vols:
    cz = m1.model.queries.center_of_mass(v, dim=3)[2]
    (footings if cz < 0.5 else columns if cz < 3.0 else beams).append(v)

assert len(footings) == 3, f"expected 3 footings, got {len(footings)}: {footings}"
assert len(columns)  == 3, f"expected 3 columns, got {len(columns)}: {columns}"
assert len(beams)    == 1, f"expected 1 beam, got {len(beams)}: {beams}"
print(f"Footings: {footings} | Columns: {columns} | Beam: {beams}")

# --- Volume physical groups (one per material) ---
m1.physical.add_volume(footings, name="Footings")
m1.physical.add_volume(columns,  name="Columns")
m1.physical.add_volume(beams,    name="Beam")

# --- Surface physical groups: bottom of footings (BC) and top of beam (load) ---
zmin = min(m1.model.queries.bounding_box(v, dim=3)[2] for v in footings)
zmax = max(m1.model.queries.bounding_box(v, dim=3)[5] for v in beams)

base_faces = m1.model.queries.select(
    "Footings", dim=2, on={'z': zmin}, tol=1e-3,
).tags()
top_faces = m1.model.queries.select(
    "Beam", dim=2, on={'z': zmax}, tol=1e-3,
).tags()

assert base_faces, f"no faces found on z={zmin} for Footings"
assert top_faces,  f"no faces found on z={zmax} for Beam"

m1.physical.add_surface(base_faces, name="Base")
m1.physical.add_surface(top_faces,  name="Top")
print(f"Base faces (z={zmin}): {base_faces}")
print(f"Top  faces (z={zmax}): {top_faces}")

# --- Mesh the now-conformal model ---
(m1.mesh.sizing
    .set_size_sources(from_points=False)
    .set_global_size(0.3))                        # 30 cm target

m1.mesh.generation.generate(dim=3)
m1.mesh.partitioning.renumber(dim=3, method='rcm', base=1)
m1.mesh.viewer()

In [ ]:
# --- 3 concrete materials (ACI: E = 4700*sqrt(f'c[MPa]) * 1e6 Pa) ---
import math
def E_c(fc_MPa: float) -> float:
    return 4700.0 * math.sqrt(fc_MPa) * 1e6

# --- FEM snapshot + apeSees bridge (named `apes`, NOT `ops`, so the
#     raw `import openseespy.opensees as ops` stays free for the
#     downstream live-domain queries in the next cell) ---
fem = m1.mesh.queries.get_fem_data(dim=3)
apes = apeSees(fem)
apes.model(ndm=3, ndf=3)

c_footing = apes.nDMaterial.ElasticIsotropic(E=E_c(21), nu=0.20, rho=2400.0)
c_column  = apes.nDMaterial.ElasticIsotropic(E=E_c(28), nu=0.20, rho=2400.0)
c_beam    = apes.nDMaterial.ElasticIsotropic(E=E_c(35), nu=0.20, rho=2400.0)

# --- Element assignments ---
apes.element.FourNodeTetrahedron(pg="Footings", material=c_footing)
apes.element.FourNodeTetrahedron(pg="Columns",  material=c_column)
apes.element.FourNodeTetrahedron(pg="Beam",     material=c_beam)

# --- Boundary conditions: pin every node on the footing base ---
apes.fix(pg="Base", dofs=(1, 1, 1))

# --- Top pressure: 100 kPa pushing down (normal=True, +mag pushes into face).
#     This session declaration stays as-is: it resolves into the FEMData
#     broker and is persisted into model.h5's neutral zone for the
#     viewer / Results. ---
with m1.loads.pattern("dead"):
    m1.loads.surface("Top", magnitude=1.0e5, normal=True)

# --- NOTE: apeSees has NO ingest and NO auto-resolution. It does NOT
#     read m1.loads / m1.masses / m1.constraints. Loads must be
#     re-declared explicitly on `apes`.
#
#     The `dead` pattern above is a *surface pressure* reduced to
#     tributary-area-weighted per-node forces. apeSees exposes no
#     surface-pressure pattern verb, and `p.load(pg=..., forces=...)`
#     applies the SAME force tuple to every node in the group (NOT the
#     per-node tributary weighting a pressure needs), so there is no
#     faithful one-line re-declaration. Re-applying this load to the
#     runnable deck is left as explicit follow-up work rather than a
#     physics-changing approximation:
# TODO(apeSees migration): m1.opensees.ingest.loads(fem) — re-declare the
#     "dead" 100 kPa top-surface pressure as explicit apeSees loads
#     (apeSees has no surface-pressure verb / no ingest; `p.load(pg=)`
#     is uniform-per-node, not tributary-weighted). The session
#     declaration above still reaches model.h5 for the viewer/Results.

# --- Populate the live OpenSees domain (faithful equivalent of the old
#     m1.opensees.build(): drives the in-process LiveOpsEmitter through
#     the full deck; does NOT call analyze — the next cell does that). ---
apes.run()
print(apes)

In [ ]:
# --- Static linear analysis ---
ops.system("BandGeneral")
ops.numberer("RCM")
ops.constraints("Plain")
ops.integrator("LoadControl", 1.0)
ops.test("NormDispIncr", 1e-8, 30)
ops.algorithm("Linear")
ops.analysis("Static")
ok = ops.analyze(1)
assert ok == 0, f"analysis failed (return {ok})"

# --- Displacements ---
node_ids = np.asarray([int(n) for n in fem.nodes.ids])
disp = np.array([ops.nodeDisp(int(n)) for n in node_ids])     # (N, 3)
print(f"Nodes: {len(node_ids)}")
print(f"Max |u|        : {np.linalg.norm(disp, axis=1).max():.3e} m")
print(f"Max u_z (down) : {disp[:, 2].min():.3e} m")

# --- Stress / strain (FourNodeTetrahedron has 1 integration point) ---
elem_ids = np.asarray([int(e) for e in fem.elements.ids])
stresses = np.array([ops.eleResponse(int(e), "stresses") for e in elem_ids])  # (E, 6)
strains  = np.array([ops.eleResponse(int(e), "strains")  for e in elem_ids])  # (E, 6)

sxx, syy, szz, sxy, syz, sxz = stresses.T
svm = np.sqrt(0.5 * ((sxx - syy)**2 + (syy - szz)**2 + (szz - sxx)**2
                     + 6 * (sxy**2 + syz**2 + sxz**2)))
print(f"Elements       : {len(elem_ids)}")
print(f"Max von Mises  : {svm.max():.3e} Pa")
print(f"Max sigma_zz   : {szz.max():.3e} Pa  | min: {szz.min():.3e} Pa")
print(f"Max eps_zz     : {strains[:, 2].max():.3e}  | min: {strains[:, 2].min():.3e}")